<a href="https://colab.research.google.com/github/goutham3010/Hospital-Patient-Flow-and-Resource-Demand-Prediction-System/blob/main/05_Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL PATIENT FLOW & DEMAND PREDICTION SYSTEM
## NOTEBOOK 05: MODEL EVALUATION & ERROR DIAGNOSTICS
**Workflow Phase 08** of the 17-Stage Hospital Operations Decision Support Architecture.

### Objectives:
1. Rigorously evaluate prediction errors across standard operational metrics:
   - **MAE** (Mean Absolute Error)
   - **RMSE** (Root Mean Squared Error)
   - **R² Score** (Coefficient of Determination)
   - **MAPE** (Mean Absolute Percentage Error)
2. Plot Actual vs. Predicted daily patient admission trajectories over the 364 unseen days of 2024.
3. Conduct residual error normality diagnostics.
4. Assess departmental ward prediction fidelity (General, ICU, HDU, NICU).


In [ ]:
# 1. IMPORT LIBRARIES & EVALUATION ARTIFACTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

sns.set_theme(style="whitegrid")
test_results = pd.read_csv('data/test_predictions.csv')
with open('models/metrics_summary.json', 'r') as f:
    metrics_summary = json.load(f)

print("Evaluation datasets and metrics loaded.")


In [ ]:
# 2. COMPARATIVE PERFORMANCE LEADERBOARD
eval_table = pd.DataFrame(metrics_summary).T
print("=== MODEL BENCHMARK LEADERBOARD (2024 UNSEEN TEST PERIOD) ===")
eval_table


In [ ]:
# 3. ACTUAL VS. PREDICTED ADMISSIONS TRAJECTORY (2024 HORIZON)
plt.figure(figsize=(15, 5))
plt.plot(pd.to_datetime(test_results['date']), test_results['actual_admissions'],
         label='Actual Hospital Admissions', color='#94a3b8', alpha=0.75, linewidth=1.2)
plt.plot(pd.to_datetime(test_results['date']), test_results['predicted_admissions'],
         label='Model Forecast (Tuned Ensemble)', color='#0284c7', linewidth=2.0)
plt.title("Daily Patient Inflow: Actual vs. Forecasted Demand (364 Days Out-of-Sample)", fontsize=13, fontweight='bold')
plt.xlabel("Date", fontsize=11)
plt.ylabel("Total Admissions", fontsize=11)
plt.legend(loc='upper right', frameon=True)
plt.show()


In [ ]:
# 4. RESIDUAL ERROR DISTRIBUTION & NORMALITY
plt.figure(figsize=(10, 4.5))
sns.histplot(test_results['residual'], kde=True, color='#0284c7', bins=30)
plt.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero Error Mean')
plt.title("Forecast Residual Error Distribution (Predicted - Actual)", fontsize=13, fontweight='bold')
plt.xlabel("Residual Error (Patients)", fontsize=11)
plt.ylabel("Frequency", fontsize=11)
plt.legend()
plt.show()

print(f"Mean Residual: {test_results['residual'].mean():.3f} (Verifies Unbiased Forecasting)")
print(f"Std Deviation: {test_results['residual'].std():.3f}")


In [ ]:
# 5. DEPARTMENTAL WARD ACCURACY
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
wards = [('general', 'General Ward', '#06b6d4'), ('icu', 'ICU', '#ef4444'),
         ('hdu', 'HDU', '#f59e0b'), ('nicu', 'NICU', '#a855f7')]

for ax, (w_key, w_name, w_col) in zip(axes.flatten(), wards):
    ax.scatter(test_results[f'actual_{w_key}'], test_results[f'pred_{w_key}'], alpha=0.5, color=w_col)
    max_val = max(test_results[f'actual_{w_key}'].max(), test_results[f'pred_{w_key}'].max())
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.7)
    ax.set_title(f"{w_name} Demand: Actual vs. Predicted", fontweight='bold')
    ax.set_xlabel("Actual Inflow")
    ax.set_ylabel("Predicted Inflow")

plt.tight_layout()
plt.show()
